# Sequence-model training guide (ResMLP / GRU / Residual-GRU)

This guide trains the windowed disturbance-force models on the same data format as the
SimpleNet guide. A short time window of recent states is fed to the network.

**Choose the architecture in the config file** `learning/config/sequence_net/model_config.yaml`
by setting `model_type` to one of:
- `resmlp`  : residual MLP on the flattened window
- `gru`     : plain stacked GRU over the window
- `res_gru` : stacked GRU with a residual skip between layers (best in the exploration)

Window length, batch size, epochs, learning rate, device live in
`learning/config/sequence_net/{data_factory_config,trainer_config}.yaml`.
Training runs on GPU automatically if available (GRUs are much faster on GPU).

In [ ]:
import os
import sys
import yaml
import matplotlib.pyplot as plt

notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import training_manager

current_dir = os.getcwd()
training_subfolder = "wind_free_space_training_cfd_our_proposal"
validation_subfolder = "wind_free_space_training_cfd_our_proposal"


def generate_data_list(subfolder):
    folder_path = os.path.join(os.path.dirname(current_dir), "data", "training", subfolder)
    file_names = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    return [os.path.join(subfolder, f) for f in file_names]


data_list = generate_data_list(training_subfolder)
validation_list = generate_data_list(validation_subfolder)
print("Training Data List:")
for data in data_list:
    print(data)

path1 = os.path.join(training_subfolder, "input_label_map_disturbance_force_label.yaml")

# architecture is chosen in learning/config/sequence_net/model_config.yaml; read it for naming
model_config_path = os.path.join(current_dir, "config", "sequence_net", "model_config.yaml")
with open(model_config_path) as f:
    model_type = yaml.safe_load(f)["model_type"]
print("\nmodel_type (from config):", model_type)

- The first run on a dataset generates `normalization_params.yaml` (per-feature standardization).
- The trainer pools all time windows and does full-minibatch Adam with early stopping.

In [ ]:
factory = training_manager.PipelineFactory(training_manager.ModelArchitecture.SEQUENCE_NET)
manager = factory.make_training_pipeline()
manager.set_up(
    training_data_menu=data_list,
    validation_data_menu=validation_list,
    input_label_map_file=path1,
    can_inspect_data=False,
)
manager.train()

plt.show()

Plot the training / validation loss curves.

In [ ]:
manager.show_result_only()
plt.show()

Save the trained model (named by subfolder + model type).

In [ ]:
model_name = training_subfolder + "_" + model_type
manager.save_model(model_name)

Test the saved model: reports per-condition RMSE (physical units, N) and MSE vs the
zero-prediction baseline.

In [ ]:
%matplotlib inline
model_name = training_subfolder + "_" + model_type
test_list = generate_data_list(validation_subfolder)

test_manager = factory.make_test_pipeline()
test_manager.set_up(
    data_menu=test_list,
    input_label_map_file=path1,
    model_name=model_name,
)
test_manager.test()
plt.show()